<div style="border-left:14px solid #708090;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#708090;letter-spacing:.08em">WELCOME</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Start here</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">How this session works, and a check that your setup runs.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 1 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

# Computational Approaches to Cultural Heritage Image Collections

This workshop is about finding, reading and working with digitised cultural heritage **programmatically**. Drawing on the idea of *distant reading*, it asks how algorithms can "read" collections of images and their descriptions at scale, and what they miss.

The path through the workshop:

1. **The IIIF standards** (notebook 1): a shared standard that lets any viewer show images from any institution, at any size and detail.
2. **The Europeana corpus** (notebook 2): an aggregator with millions of descriptions from thousands of institutions, and an API. We treat it as a *dataset*, to find out what is inside.
3. **Semantic search** (notebook 3): a model (CLIP) that lets you search images by describing them in words or with an image, and what changes when you search with a part of an image instead of the whole.
4. **A small task, then short presentations** (final task): each of you shows what you found, then we conclude.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 2 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Working in a notebook

- A notebook is a sequence of **cells**. Text cells explain; code cells run Python.
- Click a cell and press **Shift+Enter** to run it. The result appears below it.
- Run cells **in order, from the top**. Later cells use what earlier ones defined.
- Select the kernel **DH Workshop** (top right). If a cell fails with a `NameError`, run the cells above it first.
- You cannot break anything. If a notebook gets into a muddle, use **Kernel → Restart Kernel**, then run from the top.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 4 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Check your setup

Run the cell below. Each line should start with ✓. If something does not, tell the instructor now.

In [1]:
import importlib, platform, sys
from pathlib import Path

def check(label, ok, detail=""):
    print(("✓" if ok else "✗"), label, detail)

check("Python", sys.version_info >= (3, 10), platform.python_version())
for pkg in ["requests", "PIL", "ipywidgets", "numpy", "torch", "clip"]:
    try:
        importlib.import_module(pkg)
        check(f"package {pkg}", True)
    except Exception as e:
        check(f"package {pkg}", False, f"({type(e).__name__})")

import requests
for name, url in [("IIIF server (iiif.io)", "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"),
                  ("Europeana API", "https://api.europeana.eu/record/v2/search.json?wskey=api2demo&query=*&rows=0")]:
    try:
        check(name, requests.get(url, timeout=15).ok)
    except Exception as e:
        check(name, False, f"({type(e).__name__})")

root = Path(".").resolve().parent
check("dataset folder (semantic search)", (root / "data" / "DSI-Workshop").exists(), str(root / "data" / "DSI-Workshop"))

✓ Python 3.12.3
✓ package requests 
✓ package PIL 
✓ package ipywidgets 
✓ package numpy 
✓ package torch 
✓ package clip 
✓ IIIF server (iiif.io) 
✓ Europeana API 
✓ dataset folder (semantic search) /mnt/data/000_003_Code/2026-03-02_DH-Workshop/data/DSI-Workshop


In [2]:
# Setup: run this cell first.
from iiif_helpers import *

<div style="border-left:14px solid #BC8F8F;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#BC8F8F;letter-spacing:.08em">IIIF</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Images across institutions</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430"></div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 2 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Intro: The Terapixel Panorama

The [Terapixel panorama](https://terapixelpanorama.ch/) of the Battle of Murten is an impressive demonstration of the IHiif capabilities

1. Open the panorama: <https://terapixelpanorama.ch/>
2. Open the **developer tools**: press **F12** (or **Ctrl+Shift+I**; on a Mac **Cmd+Option+I**).
4. Filter by **Img** (in Firefox: **Images**).
5. **Pan and zoom** in the panorama and watch the list fill up.
6. Click one request. The **Headers** tab shows its URL, the **Preview** tab the picture that came back.

In [5]:
url_panel("Terapixel panorama, Battle of Murten", [
    ("Terapixel panorama", "https://terapixelpanorama.ch/", "Open it, press F12, Network tab, filter by Img, then pan and zoom."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 3 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## What is IIIF?

**International Image Interoperability Framework**

- A shared, open **standard** for delivering image-based resources on the web (and audio, video and 3D), so that they work across institutions.
- Also an open, global **community** of libraries, archives, museums, universities, developers and researchers who develop the specifications, build them into software and publish their collections.
- In short: images and their descriptions are offered through open interfaces (**APIs**), and any compatible tool can use them.



<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 4 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Why does it matter?

![IIIF connects institutions, through open APIs, to any viewer or tool](../misc/assets/img/iiif_why.svg)

- The images **stay at the institution**. Nothing is copied or re-hosted.
- **One viewer** can show objects from many places side by side.
- Anyone can compare, annotate and reuse objects, because the interfaces are the same everywhere.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 5 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Where do I find IIIF?

<img src="../misc/assets/img/iiif_logo.png" alt="IIIF logo" align="right" width="130">

On many publication platforms. Some show this **IIIF logo**, or a link called **IIIF manifest**, on the object page. Others hide it in a menu, or do not show it at all.

- [e-rara](https://www.e-rara.ch/) and [e-codices](https://www.e-codices.ch/): Swiss rare prints and manuscripts
- [e-manuscripta](https://www.e-manuscripta.ch/): manuscripts and archival material
- [Gallica](https://gallica.bnf.fr/): Bibliothèque nationale de France
- [Europeana](https://www.europeana.eu/): the European aggregator (next notebook)
- [Biblissima](https://portail.biblissima.fr/): a portal for medieval and early modern sources

<small>IIIF logo: International Image Interoperability Framework, public domain (text logo), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:International_Image_Interoperability_Framework_logo.png).</small>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 6 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## A Book of Hours with many owners

[Fragmentarium](https://fragmentarium.ms/) record F-mp8l is a *virtual reconstruction* of a Book of Hours (France, early 15th century) that was taken apart in the 1940s. Its leaves are held by many institutions: Yale, Kent, Cleveland Public Library and more.

One manifest brings them back together: **61 canvases**, each with an image that stays on its owner's server, and **16 ranges** that rebuild the structure of the book (the Hours of the Virgin, the Hours of the Holy Spirit, ...).

Open the **raw JSON** of the manifest in a browser tab (below) and unfold it: `sequences` → `canvases` → one canvas → `images` → `resource` → `service`. The next slide is the same tree, as a diagram.

In [8]:
BOH_MANIFEST = "https://fragmentarium.ms/metadata/iiif/F-mp8l/manifest.json"

url_panel("A Book of Hours, reassembled", [
    ("Fragmentarium record (reconstruction)", "https://fragmentarium.ms/overview/F-mp8l"),
    ("IIIF manifest = the raw JSON", BOH_MANIFEST, "Open it in a browser tab (Firefox shows JSON as an unfoldable tree). Or paste it into Mirador: Add resource."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 7 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

### How a manifest is nested

![IIIF structure: collection, manifest, ranges, canvases, images](../misc/assets/img/iiif_structure.svg)

- <b style="color:rosybrown">Manifest</b>: one object, one URL. In the Book of Hours: the whole reconstruction.
- <b style="color:crimson">Canvas</b>: a virtual page with width and height. List order is page order. Here: one leaf.
- <b style="color:blueviolet">Image</b>: attached to a canvas by a `painting` annotation. It can be hosted anywhere: every leaf on its owner's server.
- **Range**: a section (a table of contents) that groups canvases *without* reordering them.

In [11]:
# The same tree as in the browser, printed: keys, list lengths, the first element of each list.
boh = get_json(BOH_MANIFEST)
show_outline(boh)

manifest   (sc:Manifest)
  sequences   [1 items]
    [0]   (sc:Sequence)
      canvases   [61 items]
        [0]   (sc:Canvas)
          images   [1 items]
            [0]   (oa:Annotation)
              resource   (dctypes:Image)
                service
  structures   [16 items]
    [0]   (sc:Range)
      ranges   [5 items]


In [12]:
# An aggregator can generate a manifest for an institution that offers none: Europeana, for an item of the Museum Kunstpalast (Düsseldorf).
EUROPEANA_MANIFEST = "https://iiif.europeana.eu/presentation/2064114/item_TATJZ2EO2DPKQKYJYJ53YATT6KYU62LS/manifest"
show_outline(get_json(EUROPEANA_MANIFEST))

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 8 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Annotations

A page of a digitised book can carry more than its image. On e-rara and e-manuscripta, a page has a **text layer**: a transcription, stored separately from the image and pointing at it. That is the idea of an **annotation**: a note that is about a page (or a part of it) without changing it.

This manuscript is one example: <https://www.e-rara.ch/bes_1/content/pagetext/32643449>

**The story**

- **The woman:** [Juliane von Krüdener](https://de.wikipedia.org/wiki/Juliane_von_Kr%C3%BCdener) (1764–1824), born in Riga into the Baltic German nobility, married at 17 to a Russian diplomat. After a religious conversion in 1804 she became a Pietist preacher and *prophetess*, travelled through south-west Germany, Alsace and Switzerland, and gained influence even over Tsar Alexander I. A noblewoman preaching in public: this shocked her contemporaries.
- **Her fate:** she was expelled from one state after another (Württemberg, Baden, Basel) and ended up in Russia, where she died in 1824 in Crimea. Some thought the treatment too harsh.
- **The man:** Friedrich von Dillenburg defends her in this manuscript.
- **Why it is here:** the defence is *text about a person*, preserved with the page images. Reading it takes a human. What would a program do with it?


In [14]:
url_panel("Text layers on e-rara and e-manuscripta", [
    ("The annotated manuscript (page text, e-rara)", "https://www.e-rara.ch/bes_1/content/pagetext/32643449", "The transcription of the page, separate from the image."),
    ("Another page text: a handwritten note by Emmi Ball-Hennings (e-manuscripta, Swiss National Library)", "https://www.e-manuscripta.ch/snl/content/pagetext/4317519", "Open it to see the transcription as plain text."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 9 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How it works behind the scenes

Two specifications do the work, and each has its own kind of URL.

| API | Question it answers | What you get |
|---|---|---|
| **Presentation API** | What is this object, and how is it organised? | A JSON *manifest* |
| **Image API** | Give me this image, in this form. | Pixels, selected by a URL |

Below: the manifest of a small book (five pages, BnF / Gallica, from the IIIF Cookbook), and the URL of one image from it. Open both in a browser tab. The manifest contains the base URL of the image server; the image URL is built from it.


In [15]:
BOOK_URL = "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"
book = get_json(BOOK_URL)
PAGE = book["items"][2]                   # the title page
_, SERVICE = canvas_image(PAGE)
info = get_json(f"{SERVICE}/info.json")   # what the server can do
W, H = info["width"], info["height"]

url_panel("One object, one image", [
    ("Presentation API: the manifest (JSON)", BOOK_URL, "Open it in a browser tab: title, canvases in order, and for each the address of its image."),
    ("Image API: the title page as an image", iiif_url(SERVICE, size="600,"), "The counterpart: pixels, delivered by a different server."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 10 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One image, any view

An image service answers URLs of a fixed form:

```
{service}/{region}/{size}/{rotation}/{quality}.{format}
```

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

In [16]:
show_views(SERVICE, [
    ("full", "full", "260,", "0", "default"),
    ("a region", f"{W*28//100},{H*45//100},{W*33//100},{H*23//100}", "260,", "0", "default"),
    ("rotated 90°", "full", ",260", "90", "default"),
    ("grey", "full", "260,", "0", "gray"),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 11 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Changing how an image is displayed by editing its URL

*Optional, in your own browser, at your own pace.*

Every view of an image is just a URL. Open the image below in a browser tab, then change one segment of the address in the address bar and press Enter.

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

- <b style="color:rosybrown">region</b>: `full`, `square`, `x,y,w,h` (pixels), `pct:x,y,w,h` (percent)
- <b style="color:coral">size</b>: `max`, `400,` (width), `,300` (height), `!400,300` (fit inside), `pct:50`
- <b style="color:crimson">rotation</b>: `0`, `90`, `180`, `270`, `!0` (mirrored)
- <b style="color:blueviolet">quality</b> and <b style="color:blueviolet">format</b>: `default`, `gray`, `bitonal`; `.jpg`, `.png`


In [18]:
EDIT_SERVICE = "https://apsida.cut.ac.cy/iiif-img/34769"   # image service of https://apsida.cut.ac.cy/iiif/36404/manifest
url_panel("An image from Apsida (Cyprus University of Technology)", [
    ("Image URL", iiif_url(EDIT_SERVICE, size="600,"), "Open it in a browser tab, then edit the segments in the address bar."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 12 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How do I use it?

1. Find the **manifest link** (or the IIIF logo) on the object page.
2. **Copy** the link.
3. Open a IIIF viewer, for example [Mirador](https://projectmirador.org/demo/) or [Universal Viewer](https://universalviewer.io/).
4. **Paste** the link. The object appears in the viewer.

In Mirador you can place several objects side by side to compare them, and a workspace can be exported and imported again. Many IIIF tools work with images without downloading and uploading them: for example [Tropy](https://tropy.org/) for archival material, or eScriptorium and Transkribus for text recognition. A long list: <https://iiif.io/get-started/tools/>.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 13 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Mirador: paste in a manifest

[Mirador](https://projectmirador.org/) is given only the manifest URL. It works out page order, thumbnails, deep zoom and the table of contents by itself.

**Your turn** *(about 5 minutes)*

1. Open the **[Mirador demo](https://projectmirador.org/demo/)**.
2. Click **Add resource**, paste one of the two manifest URLs below, and press **Add**. Add the second one too and place them side by side.
3. Then find a manifest of your own: open one of the sites from *Where do I find IIIF?*, pick an object, and look for the IIIF logo or a link called *IIIF* or *Manifest*, often in a share, download or export menu.

**Example manifests** *(copy the link)*

- Watercolour *0m* by Lucius Burckhardt, e-manuscripta:
  `https://www.e-manuscripta.ch/i3f/v20/4602948/manifest`
- *Der Wanderer über dem Nebelmeer*, Deutsche Fotothek via arthistoricum.net:
  `https://iiif.arthistoricum.net/proxy/fotothek/df_hauptkatalog_0182654/manifest.json`

<small>**Credits.**
*0m* / von Lucius Burckhardt, [S.l.], [between 1925 and 2003]. Watercolour on paper, 23 x 31 cm. Universitätsbibliothek Basel, UBH NL 351 : P:792 (Nachlass Lucius Burckhardt and Annemarie Burckhardt-Wackernagel). Via [e-manuscripta](https://www.e-manuscripta.ch/), DOI [10.7891/e-manuscripta-170584](https://doi.org/10.7891/e-manuscripta-170584).<br>
© Deutsche Fotothek / Karin Stein. License: all rights reserved. Higher-resolution copies, prints and usage rights can be obtained from the [Deutsche Fotothek](https://www.deutschefotothek.de/).</small>